# Quantitative comparison: Online STSP, STS-Miner, and STBFM

This notebook runs a fresh quantitative experiment on the preprocessed TSMC2014 NYC data. It compares the online STSP miner against STS-Miner and STBFM across the configured event-type counts and online significance thresholds.

For each `(event_type_count, online_theta)` pair, the online miner is run once and fresh snapshots are written at the configured checkpoints. For each matching prefix size, STS-Miner and STBFM are tuned by searching over spatial radius `R`, temporal window `T`, and a binary-searched score threshold. The selected offline run is the closest pattern-count match to the online snapshot, with the default acceptance tolerance set to 10%.

In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from datetime import datetime
from math import ceil
from pathlib import Path
import sys
from time import perf_counter
from typing import Callable

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "Source"))

from spatio_temporal_network import (
    LearningParameters,
    OnlineSTSPMiner,
    initialize_network_from_events,
)
from sts_miner import STSMiner, add_local_meter_coordinates, patterns_to_frame as sts_patterns_to_frame
from stbfm import STBFM, patterns_to_frame as stbfm_patterns_to_frame

## Configuration

`MAX_PATTERN_LENGTH` limits sequence expansion in all three algorithms. Set it to `None` for exhaustive sequence expansion, but expect substantially longer runtimes.

In [2]:
DATA_PATH = PROJECT_ROOT / "Data" / "Preprocessed" / "dataset_TSMC2014_NYC_preprocessed.csv"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "Quantitative_experiment"
RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_ROOT / RUN_ID

EVENT_TYPE_COUNTS = (15,)
MAX_EVENTS = 10000
CHECKPOINT_EVERY = 500
CHECKPOINTS = tuple(range(CHECKPOINT_EVERY, MAX_EVENTS + 1, CHECKPOINT_EVERY))

GRID_SHAPE = (10, 10)
ONLINE_SPATIAL_THRESHOLD_METERS = 10_000.0
ONLINE_ETA = 1.0
ONLINE_TAU_ACTIVATION = 120.0
ONLINE_TAU_REFRACTORY = 12.0
ONLINE_THETAS = (0.1, 0.15, 0.2)
ONLINE_LAMBDA_DECAY = 0.1

MAX_PATTERN_LENGTH = 2
MAX_SNAPSHOT_PATTERNS = None
RELATIVE_PATTERN_COUNT_TOLERANCE = 0.1

R_CANDIDATES_METERS = (250.0, 500.0, 1_000.0, 2_000.0, 5_000.0, 10_000.0)
T_CANDIDATES_MINUTES = (15.0,30.0,60.0,120.0,240.0)
BINARY_SEARCH_ITERATIONS = 8
STOP_AFTER_FIRST_TOLERANCE_MATCH = True
STS_THRESHOLD_LOW = 1.0
STS_THRESHOLD_HIGH_LIMIT = 1_000_000.0
STBFM_THRESHOLD_LOW = 0.0
STBFM_THRESHOLD_HIGH = 1.0

RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR

PosixPath('/Users/piotr/Praca/Nauka/publikacje/SNNs_ST_Patterns/Experiments_software/Results/Quantitative_experiment/run_20260601_125554')

In [3]:
def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def pattern_count_tolerance(target_count: int) -> int:
    if target_count == 0:
        return 0
    return max(1, ceil(target_count * RELATIVE_PATTERN_COUNT_TOLERANCE))


def within_tolerance(pattern_count: int, target_count: int) -> bool:
    return abs(pattern_count - target_count) <= pattern_count_tolerance(target_count)


def load_selected_events(event_type_count: int) -> tuple[pd.DataFrame, tuple[str, ...]]:
    raw_events = pd.read_csv(DATA_PATH).sort_values("timestamp", ignore_index=True)
    selected_types = tuple(raw_events["Event_type"].value_counts().head(event_type_count).index)
    selected_events = raw_events[raw_events["Event_type"].isin(selected_types)].head(MAX_EVENTS).copy()
    selected_events = selected_events.sort_values("timestamp", ignore_index=True)
    if len(selected_events) < MAX_EVENTS:
        raise ValueError(
            f"Only {len(selected_events)} events are available for the top {event_type_count} event types."
        )
    return selected_events, selected_types


def make_online_parameters(events: pd.DataFrame, online_theta: float) -> LearningParameters:
    observation_window_minutes = float(events["timestamp"].iloc[-1] - events["timestamp"].iloc[0])
    if observation_window_minutes <= 0:
        raise ValueError("The selected stream must cover a positive time span.")
    return LearningParameters(
        eta=ONLINE_ETA,
        lambda_decay=ONLINE_LAMBDA_DECAY,
        tau_activation=ONLINE_TAU_ACTIVATION,
        tau_refractory=ONLINE_TAU_REFRACTORY,
        theta=online_theta,
    )

In [4]:
def run_online_snapshots(events: pd.DataFrame, output_dir: Path, online_theta: float) -> pd.DataFrame:
    online_dir = output_dir / "online"
    online_dir.mkdir(parents=True, exist_ok=True)

    parameters = make_online_parameters(events, online_theta)
    network = initialize_network_from_events(
        events,
        grid_shape=GRID_SHAPE,
        spatial_threshold=ONLINE_SPATIAL_THRESHOLD_METERS,
    )
    miner = OnlineSTSPMiner(network, parameters)

    rows = []
    start_time = perf_counter()
    previous_checkpoint_time = start_time
    checkpoint_set = set(CHECKPOINTS)

    for event_index, event in enumerate(events.itertuples(index=False), start=1):
        miner.process_event(pd.Series(event._asdict()), extract_patterns=False)
        if event_index not in checkpoint_set:
            continue

        extraction_start = perf_counter()
        patterns = miner.extract_patterns(
            max_path_length=MAX_PATTERN_LENGTH,
            max_patterns=MAX_SNAPSHOT_PATTERNS,
        )
        extraction_seconds = perf_counter() - extraction_start
        now = perf_counter()

        stem = f"online_snapshot_{event_index:06d}"
        patterns_path = online_dir / f"{stem}_patterns.csv"
        metadata_path = online_dir / f"{stem}_metadata.json"
        miner.patterns_frame(patterns).to_csv(patterns_path, index=False)

        row = {
            "algorithm": "online_stsp",
            "online_theta": online_theta,
            "event_count": event_index,
            "pattern_count": len(patterns),
            "significant_synapse_count": len(miner.significant_synapses),
            "interval_seconds": now - previous_checkpoint_time,
            "cumulative_seconds": now - start_time,
            "snapshot_extraction_seconds": extraction_seconds,
            "patterns_path": str(patterns_path.relative_to(PROJECT_ROOT)),
            "metadata_path": str(metadata_path.relative_to(PROJECT_ROOT)),
        }
        metadata = {
            **row,
            "network": network.summary(),
            "learning": {
                "eta": parameters.eta,
                "lambda_decay": parameters.lambda_decay,
                "tau_activation": parameters.tau_activation,
                "tau_refractory": parameters.tau_refractory,
                "theta": parameters.theta,
                "max_weight": parameters.max_weight,
                "significance_threshold": parameters.significance_threshold,
            },
            "max_pattern_length": MAX_PATTERN_LENGTH,
            "max_snapshot_patterns": MAX_SNAPSHOT_PATTERNS,
            "online_theta": online_theta,
        }
        write_json(metadata_path, metadata)
        rows.append(row)
        previous_checkpoint_time = now

    frame = pd.DataFrame(rows)
    frame.to_csv(online_dir / "online_summary.csv", index=False)
    return frame

In [5]:
@dataclass
class TrialResult:
    algorithm: str
    event_count: int
    spatial_radius: float
    temporal_window: float
    threshold: float
    pattern_count: int
    mining_seconds: float
    patterns: list


def run_sts_trial(events: pd.DataFrame, spatial_radius: float, temporal_window: float, threshold: float) -> tuple[list, float]:
    start = perf_counter()
    miner = STSMiner(
        events,
        spatial_radius=spatial_radius,
        temporal_window=temporal_window,
        min_sequence_index=threshold,
        max_length=MAX_PATTERN_LENGTH,
    )
    patterns = miner.mine()
    return patterns, perf_counter() - start


def run_stbfm_trial(events: pd.DataFrame, spatial_radius: float, temporal_window: float, threshold: float) -> tuple[list, float]:
    start = perf_counter()
    miner = STBFM(
        events,
        spatial_radius=spatial_radius,
        temporal_window=temporal_window,
        min_participation_index=threshold,
        max_length=MAX_PATTERN_LENGTH,
        threshold_inclusive=False,
    )
    patterns = miner.mine()
    return patterns, perf_counter() - start


def score_trial(trial: TrialResult, target_count: int) -> tuple[int, float, float, float]:
    return (
        abs(trial.pattern_count - target_count),
        trial.mining_seconds,
        trial.spatial_radius,
        trial.temporal_window,
    )

In [6]:
def tune_threshold_for_radius_window(
    *,
    algorithm: str,
    events: pd.DataFrame,
    event_count: int,
    target_count: int,
    spatial_radius: float,
    temporal_window: float,
    run_trial: Callable[[pd.DataFrame, float, float, float], tuple[list, float]],
    threshold_low: float,
    threshold_high: float,
    expandable_high: bool,
) -> tuple[TrialResult, list[dict]]:
    attempts = []

    def evaluate(threshold: float, phase: str) -> TrialResult:
        patterns, seconds = run_trial(events, spatial_radius, temporal_window, threshold)
        trial = TrialResult(
            algorithm=algorithm,
            event_count=event_count,
            spatial_radius=spatial_radius,
            temporal_window=temporal_window,
            threshold=threshold,
            pattern_count=len(patterns),
            mining_seconds=seconds,
            patterns=patterns,
        )
        attempts.append({
            "algorithm": algorithm,
            "event_count": event_count,
            "target_pattern_count": target_count,
            "spatial_radius": spatial_radius,
            "temporal_window": temporal_window,
            "threshold": threshold,
            "pattern_count": trial.pattern_count,
            "mining_seconds": trial.mining_seconds,
            "phase": phase,
        })
        return trial

    low_trial = evaluate(threshold_low, "low")
    best = low_trial
    low = threshold_low
    high = threshold_high

    high_trial = evaluate(high, "high")
    if score_trial(high_trial, target_count) < score_trial(best, target_count):
        best = high_trial

    while expandable_high and high_trial.pattern_count > target_count and high < STS_THRESHOLD_HIGH_LIMIT:
        low = high
        high = min(high * 2.0, STS_THRESHOLD_HIGH_LIMIT)
        high_trial = evaluate(high, "expand_high")
        if score_trial(high_trial, target_count) < score_trial(best, target_count):
            best = high_trial

    for iteration in range(BINARY_SEARCH_ITERATIONS):
        midpoint = (low + high) / 2.0
        trial = evaluate(midpoint, f"binary_{iteration + 1}")
        if score_trial(trial, target_count) < score_trial(best, target_count):
            best = trial

        if trial.pattern_count > target_count:
            low = midpoint
        else:
            high = midpoint

        if within_tolerance(trial.pattern_count, target_count):
            break

    return best, attempts

In [7]:
def tune_offline_algorithm(
    *,
    algorithm: str,
    events: pd.DataFrame,
    event_count: int,
    target_count: int,
    output_dir: Path,
    online_theta: float,
) -> dict:
    if algorithm == "sts_miner":
        run_trial = run_sts_trial
        threshold_low = STS_THRESHOLD_LOW
        threshold_high = max(STS_THRESHOLD_LOW * 2.0, 2.0)
        expandable_high = True
        to_frame = sts_patterns_to_frame
    elif algorithm == "stbfm":
        run_trial = run_stbfm_trial
        threshold_low = STBFM_THRESHOLD_LOW
        threshold_high = STBFM_THRESHOLD_HIGH
        expandable_high = False
        to_frame = stbfm_patterns_to_frame
    else:
        raise ValueError(f"Unsupported algorithm: {algorithm}")

    all_attempts = []
    best: TrialResult | None = None
    for spatial_radius in R_CANDIDATES_METERS:
        for temporal_window in T_CANDIDATES_MINUTES:
            candidate, attempts = tune_threshold_for_radius_window(
                algorithm=algorithm,
                events=events,
                event_count=event_count,
                target_count=target_count,
                spatial_radius=spatial_radius,
                temporal_window=temporal_window,
                run_trial=run_trial,
                threshold_low=threshold_low,
                threshold_high=threshold_high,
                expandable_high=expandable_high,
            )
            all_attempts.extend(attempts)
            if best is None or score_trial(candidate, target_count) < score_trial(best, target_count):
                best = candidate
            if STOP_AFTER_FIRST_TOLERANCE_MATCH and best is not None and within_tolerance(best.pattern_count, target_count):
                break
        if STOP_AFTER_FIRST_TOLERANCE_MATCH and best is not None and within_tolerance(best.pattern_count, target_count):
            break

    if best is None:
        raise RuntimeError(f"No trial was run for {algorithm} at {event_count} events.")

    algorithm_dir = output_dir / algorithm
    algorithm_dir.mkdir(parents=True, exist_ok=True)
    stem = f"{algorithm}_{event_count:06d}"
    patterns_path = algorithm_dir / f"{stem}_best_patterns.csv"
    metadata_path = algorithm_dir / f"{stem}_metadata.json"
    attempts_path = algorithm_dir / f"{stem}_attempts.csv"

    to_frame(best.patterns).to_csv(patterns_path, index=False)
    pd.DataFrame(all_attempts).to_csv(attempts_path, index=False)

    summary = {
        "algorithm": algorithm,
        "online_theta": online_theta,
        "event_count": event_count,
        "target_pattern_count": target_count,
        "pattern_count": best.pattern_count,
        "absolute_pattern_count_difference": abs(best.pattern_count - target_count),
        "relative_pattern_count_difference": None if target_count == 0 else abs(best.pattern_count - target_count) / target_count,
        "within_tolerance": within_tolerance(best.pattern_count, target_count),
        "tolerance_patterns": pattern_count_tolerance(target_count),
        "spatial_radius": best.spatial_radius,
        "temporal_window": best.temporal_window,
        "threshold": best.threshold,
        "mining_seconds": best.mining_seconds,
        "attempt_count": len(all_attempts),
        "patterns_path": str(patterns_path.relative_to(PROJECT_ROOT)),
        "metadata_path": str(metadata_path.relative_to(PROJECT_ROOT)),
        "attempts_path": str(attempts_path.relative_to(PROJECT_ROOT)),
    }
    write_json(metadata_path, summary)
    return summary

## Run the experiment

This cell performs the full experiment. It may take a long time because every offline checkpoint evaluates multiple `(R, T, threshold)` trials.

In [8]:
experiment_start = perf_counter()
all_summary_rows = []

experiment_config = {
    "run_id": RUN_ID,
    "data_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "event_type_counts": EVENT_TYPE_COUNTS,
    "online_thetas": ONLINE_THETAS,
    "max_events": MAX_EVENTS,
    "checkpoints": CHECKPOINTS,
    "grid_shape": GRID_SHAPE,
    "online_spatial_threshold_meters": ONLINE_SPATIAL_THRESHOLD_METERS,
    "online_eta": ONLINE_ETA,
    "online_tau_activation": ONLINE_TAU_ACTIVATION,
    "online_tau_refractory": ONLINE_TAU_REFRACTORY,
    "online_lambda_decay": ONLINE_LAMBDA_DECAY,
    "max_pattern_length": MAX_PATTERN_LENGTH,
    "max_snapshot_patterns": MAX_SNAPSHOT_PATTERNS,
    "relative_pattern_count_tolerance": RELATIVE_PATTERN_COUNT_TOLERANCE,
    "r_candidates_meters": R_CANDIDATES_METERS,
    "t_candidates_minutes": T_CANDIDATES_MINUTES,
    "binary_search_iterations": BINARY_SEARCH_ITERATIONS,
    "stop_after_first_tolerance_match": STOP_AFTER_FIRST_TOLERANCE_MATCH,
}
write_json(RUN_DIR / "experiment_config.json", experiment_config)

for event_type_count in EVENT_TYPE_COUNTS:
    print(f"Running event_type_count={event_type_count}")
    type_dir = RUN_DIR / f"event_types_{event_type_count:02d}"
    type_dir.mkdir(parents=True, exist_ok=True)

    selected_events, selected_types = load_selected_events(event_type_count)
    selected_events.to_csv(type_dir / "selected_events.csv", index=False)
    write_json(
        type_dir / "selected_event_types.json",
        {"event_type_count": event_type_count, "event_types": selected_types},
    )

    for online_theta in ONLINE_THETAS:
        theta_label = f"theta_{online_theta:.2f}".replace(".", "_")
        pair_dir = type_dir / theta_label
        pair_dir.mkdir(parents=True, exist_ok=True)
        print(f"  online_theta={online_theta}")

        online_summary = run_online_snapshots(selected_events, pair_dir, online_theta)
        for _, online_row in online_summary.iterrows():
            event_count = int(online_row["event_count"])
            target_count = int(online_row["pattern_count"])
            prefix_events = add_local_meter_coordinates(selected_events.head(event_count).copy())

            online_result = {
                "event_type_count": event_type_count,
                "online_theta": online_theta,
                "algorithm": "online_stsp",
                "event_count": event_count,
                "target_pattern_count": target_count,
                "pattern_count": target_count,
                "within_tolerance": True,
                "mining_seconds": float(online_row["interval_seconds"]),
                "cumulative_seconds": float(online_row["cumulative_seconds"]),
                "snapshot_extraction_seconds": float(online_row["snapshot_extraction_seconds"]),
                "patterns_path": online_row["patterns_path"],
                "metadata_path": online_row["metadata_path"],
            }
            all_summary_rows.append(online_result)

            for algorithm in ("sts_miner", "stbfm"):
                print(f"    {algorithm}: event_count={event_count}, target={target_count}")
                offline_summary = tune_offline_algorithm(
                    algorithm=algorithm,
                    events=prefix_events,
                    event_count=event_count,
                    target_count=target_count,
                    output_dir=pair_dir,
                    online_theta=online_theta,
                )
                offline_summary["event_type_count"] = event_type_count
                all_summary_rows.append(offline_summary)

summary = pd.DataFrame(all_summary_rows)
summary.to_csv(RUN_DIR / "summary.csv", index=False)
write_json(
    RUN_DIR / "experiment_metadata.json",
    {"run_id": RUN_ID, "elapsed_seconds": perf_counter() - experiment_start, "summary_path": "summary.csv"},
)
summary
 

Running event_type_count=15
  online_theta=0.1
    sts_miner: event_count=500, target=28
    stbfm: event_count=500, target=28
    sts_miner: event_count=1000, target=354
    stbfm: event_count=1000, target=354
    sts_miner: event_count=1500, target=228
    stbfm: event_count=1500, target=228
    sts_miner: event_count=2000, target=418
    stbfm: event_count=2000, target=418
    sts_miner: event_count=2500, target=15
    stbfm: event_count=2500, target=15
    sts_miner: event_count=3000, target=111
    stbfm: event_count=3000, target=111
    sts_miner: event_count=3500, target=258
    stbfm: event_count=3500, target=258
    sts_miner: event_count=4000, target=32
    stbfm: event_count=4000, target=32
    sts_miner: event_count=4500, target=470
    stbfm: event_count=4500, target=470
    sts_miner: event_count=5000, target=224
    stbfm: event_count=5000, target=224
    sts_miner: event_count=5500, target=247
    stbfm: event_count=5500, target=247
    sts_miner: event_count=6000, targ

KeyboardInterrupt: 